# 01 — Data audit

First look at both raw files, exactly as stored. Nothing is cleaned or
dropped here; the point is to measure every defect so that Chapter 2 can
handle each one deliberately. All counts printed below are written to
`report/stats/ch01_audit.json` and quoted in the report from there.

**Datasets**
- `movies_eda_checkpoint_2.csv` — primary, scraped from the TMDB API (has budget/revenue)
- `top10K-TMDB-movies.csv` — secondary, from Kaggle (`ahsanaseer/top-rated-tmdb-movies-10k`)

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.load import load_primary, load_secondary
from src.reporting import fmt_int, write_stats, write_table

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

stats = {}

## Primary dataset, as stored

Loaded with `zero_to_nan=False`: budgets and revenues appear exactly as
TMDB serves them, zeros included.

In [2]:
df = load_primary(zero_to_nan=False)
stats["primary_rows"], stats["primary_cols"] = df.shape
df.shape

(10000, 17)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Title               10000 non-null  str           
 1   Release_Date        9997 non-null   datetime64[us]
 2   Release_Year        9997 non-null   Int64         
 3   Budget              10000 non-null  int64         
 4   Worldwide_Revenue   10000 non-null  int64         
 5   Runtime_Mins        10000 non-null  int64         
 6   Genres              9998 non-null   str           
 7   Content_Rating      10000 non-null  str           
 8   Audience_Rating     10000 non-null  float64       
 9   Total_Votes         10000 non-null  int64         
 10  Popularity_Score    10000 non-null  float64       
 11  Original_Language   10000 non-null  str           
 12  Director            10000 non-null  str           
 13  Lead_Actors         10000 non-null  str           
 14  Pr

In [4]:
df.head()

,Title,Release_Date,Release_Year,Budget,Worldwide_Revenue,Runtime_Mins,Genres,Content_Rating,Audience_Rating,Total_Votes,Popularity_Score,Original_Language,Director,Lead_Actors,Production_Company,Production_Country,Overview
0,Interstellar,2014-11-05,2014,165000000,746606706,169,"Adventure, Drama, Science Fiction",PG-13,8.489,41294,72.2078,en,Christopher Nolan,"Matthew McConaughey, Anne Hathaway, Michael Caine","Legendary Pictures, Syncopy","United Kingdom, United States of America",The adventures of a group of explorers who mak...
1,Guardians of the Galaxy,2014-07-30,2014,170000000,772776600,121,"Action, Science Fiction, Adventure",PG-13,7.908,30360,19.0340,en,James Gunn,"Chris Pratt, Zoe Saldaña, Dave Bautista",Marvel Studios,United States of America,"Light years from Earth, 26 years after being a..."
2,Harry Potter and the Philosopher's Stone,2001-11-16,2001,125000000,1029374615,152,"Adventure, Fantasy",PG,7.904,30293,56.2691,en,Chris Columbus,"Daniel Radcliffe, Rupert Grint, Emma Watson","Warner Bros. Pictures, Heyday Films","United Kingdom, United States of America",Harry Potter has lived under the stairs at his...
3,The Shawshank Redemption,1994-09-23,1994,25000000,28341469,142,"Drama, Crime",R,8.731,31436,62.6216,en,Frank Darabont,"Tim Robbins, Morgan Freeman, Bob Gunton",Castle Rock Entertainment,United States of America,Imprisoned in the 1940s for the double murder ...
4,Avengers: Infinity War,2018-04-25,2018,300000000,2052415039,149,"Adventure, Action, Science Fiction",PG-13,8.200,33006,99.5498,en,Anthony Russo,"Robert Downey Jr., Chris Evans, Chris Hemsworth",Marvel Studios,United States of America,As the Avengers and their allies have continue...


In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Title,10000,9606,Return,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Release_Date,9997,NaN,NaN,NaN,2005-09-27 04:49:05.683705,1895-03-22 00:00:00,1998-11-13 00:00:00,2010-08-16 00:00:00,2017-11-04 00:00:00,2026-09-16 00:00:00,NaN
Release_Year,9997.0,<NA>,<NA>,<NA>,2005.20136,1895.0,1998.0,2010.0,2017.0,2026.0,17.398674
Budget,10000.0,NaN,NaN,NaN,24605346.008,0.0,0.0,9000000.0,30000000.0,658800000.0,41594289.392377
Worldwide_Revenue,10000.0,NaN,NaN,NaN,72003687.5183,0.0,95852.0,14000000.0,67071592.25,2923706026.0,168427949.56593
Runtime_Mins,10000.0,NaN,NaN,NaN,105.957,0.0,94.0,104.0,117.0,367.0,22.751279
Genres,9998,2101,Drama,542,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Content_Rating,10000,8,R,4016,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Audience_Rating,10000.0,NaN,NaN,NaN,6.662549,2.2,6.164,6.7,7.2,9.152,0.775403
Total_Votes,10000.0,NaN,NaN,NaN,2253.043,367.0,563.0,999.0,2273.0,41294.0,3461.315071


## Missing values by column

True nulls only — the zero-coded missing financials are counted separately below.
`Release_Date` is stored as `M/D/YYYY` except for two 1890s Lumière shorts
stored as ISO `YYYY-MM-DD`; the loader parses both, leaving 3 genuine nulls.

In [6]:
missing = pd.DataFrame(
    {
        "n_missing": df.isnull().sum(),
        "pct_missing": (df.isnull().mean() * 100).round(2),
    }
).sort_values("n_missing", ascending=False)
stats["primary_null_release_date"] = int(df["Release_Date"].isnull().sum())
stats["primary_null_genres"] = int(df["Genres"].isnull().sum())
stats["primary_null_production_company"] = int(df["Production_Company"].isnull().sum())
stats["primary_null_production_country"] = int(df["Production_Country"].isnull().sum())
missing

,n_missing,pct_missing
Production_Company,27,0.27
Production_Country,9,0.09
Release_Date,3,0.03
Release_Year,3,0.03
Genres,2,0.02
Overview,1,0.01
Title,0,0.00
Worldwide_Revenue,0,0.00
Budget,0,0.00
Audience_Rating,0,0.00


## Zero-as-missing in the financial columns

TMDB stores *unknown* budget or revenue as `0`, not null. Treating those
zeros as real values would poison every financial statistic, so Chapter 2
converts them to `NaN`. Here we just count them.

In [7]:
zero_budget = int((df["Budget"] == 0).sum())
zero_revenue = int((df["Worldwide_Revenue"] == 0).sum())
both_positive = int(((df["Budget"] > 0) & (df["Worldwide_Revenue"] > 0)).sum())
budget_only = int(((df["Budget"] > 0) & (df["Worldwide_Revenue"] == 0)).sum())

stats.update(
    zero_budget=zero_budget,
    zero_revenue=zero_revenue,
    both_financials_positive=both_positive,
    budget_but_zero_revenue=budget_only,
)
print(f"zero budget:               {zero_budget:>6,}")
print(f"zero revenue:              {zero_revenue:>6,}")
print(f"both budget & revenue > 0: {both_positive:>6,}")
print(f"budget > 0, revenue == 0:  {budget_only:>6,}")

zero budget:                2,814
zero revenue:               2,288
both budget & revenue > 0:  6,632
budget > 0, revenue == 0:     554


How badly would the zeros bias a naive mean? Compare the mean budget over
all rows against the mean over rows with a recorded (non-zero) budget.

In [8]:
mean_with_zeros = df["Budget"].mean()
mean_positive = df.loc[df["Budget"] > 0, "Budget"].mean()
stats["mean_budget_with_zeros"] = round(float(mean_with_zeros))
stats["mean_budget_positive_only"] = round(float(mean_positive))
print(f"mean budget incl. zeros: ${mean_with_zeros:>13,.0f}")
print(f"mean budget, budget > 0: ${mean_positive:>13,.0f}")
print(f"understatement:          {(1 - mean_with_zeros / mean_positive) * 100:.1f}%")

mean budget incl. zeros: $   24,605,346
mean budget, budget > 0: $   34,240,671
understatement:          28.1%


## Implausible values

Budgets or revenues between \$1 and \$999 are data-entry errors or figures
recorded in thousands; a zero runtime is impossible for a released film.

In [9]:
tiny_budget = int(df["Budget"].between(1, 999).sum())
tiny_revenue = int(df["Worldwide_Revenue"].between(1, 999).sum())
zero_runtime = int((df["Runtime_Mins"] == 0).sum())
stats.update(
    budget_1_to_999=tiny_budget,
    revenue_1_to_999=tiny_revenue,
    zero_runtime=zero_runtime,
    runtime_max=int(df["Runtime_Mins"].max()),
    runtime_median=float(df["Runtime_Mins"].median()),
)
print(f"budget in [1, 999]:  {tiny_budget}")
print(f"revenue in [1, 999]: {tiny_revenue}")
print(f"runtime == 0:        {zero_runtime}")
df.loc[df["Budget"].between(1, 999), ["Title", "Release_Year", "Budget", "Worldwide_Revenue"]]

budget in [1, 999]:  13
revenue in [1, 999]: 9
runtime == 0:        1


,Title,Release_Year,Budget,Worldwide_Revenue
2631,Swapped,2026,7,7
3223,Creep,2014,420,0
5656,East of Eden,1955,2,5000000
6288,The Great Train Robbery,1903,150,0
7217,The Quiet Girl,2022,1,6
7794,Cowspiracy: The Sustainability Secret,2014,117,0
8149,Rampage,2009,120,428
8596,Meshes of the Afternoon,1943,275,0
8791,Red Dog,2011,5,13
9364,Bonanza: The Return,1993,93,0


## Unreleased and near-future films

Revenue for films dated after 2025 is necessarily incomplete; Chapter 2
excludes them from all revenue and ROI analysis.

In [10]:
future = int((df["Release_Year"] > 2025).sum())
stats["future_films"] = future
stats["year_min"] = int(df["Release_Year"].min())
stats["year_max"] = int(df["Release_Year"].max())
print(f"films with Release_Year > 2025: {future}")
df.loc[df["Release_Year"] > 2025, "Release_Year"].value_counts().sort_index()

films with Release_Year > 2025: 92


Release_Year
2026    92
Name: count, dtype: Int64

## Duplicates

Exact duplicate rows are errors. Repeated titles are mostly legitimate
remakes (different years), so deduplication must key on title **plus** year.

In [11]:
exact_dups = int(df.duplicated().sum())
title_dups = int(df["Title"].duplicated().sum())
title_year_dups = int(df.duplicated(subset=["Title", "Release_Year"]).sum())
stats.update(
    exact_duplicate_rows=exact_dups,
    duplicated_titles=title_dups,
    duplicated_title_year=title_year_dups,
)
print(f"exact duplicate rows:        {exact_dups}")
print(f"repeated titles:             {title_dups}")
print(f"repeated (title, year) pairs: {title_year_dups}")

exact duplicate rows:        4
repeated titles:             394
repeated (title, year) pairs: 12


In [12]:
# A few repeated titles, to show they are remakes, not errors
rep = df[df["Title"].duplicated(keep=False)].sort_values(["Title", "Release_Year"])
rep[["Title", "Release_Year", "Director"]].head(12)

,Title,Release_Year,Director
364,12 Angry Men,1957,Sidney Lumet
8420,12 Angry Men,1997,William Friedkin
8861,3:10 to Yuma,1957,Delmer Daves
1341,3:10 to Yuma,2007,James Mangold
8886,A Christmas Carol,1984,Clive Donner
1060,A Christmas Carol,2009,Robert Zemeckis
865,A Nightmare on Elm Street,1984,Wes Craven
1939,A Nightmare on Elm Street,2010,Samuel Bayer
5869,Adventures in Babysitting,1987,Chris Columbus
8734,Adventures in Babysitting,2016,John Schultz


## Category hygiene

`Content_Rating` has redundant levels: `Unrated` and `NR` mean the same
thing, and one film carries a stray `"PG-13 "` with a trailing space.

In [13]:
cr = df["Content_Rating"].value_counts(dropna=False)
stats["content_rating_unrated"] = int(cr.get("Unrated", 0))
stats["content_rating_nr"] = int(cr.get("NR", 0))
stats["content_rating_stray_pg13"] = int((df["Content_Rating"] == "PG-13 ").sum())
cr

Content_Rating
R          4016
PG-13      2370
PG         1301
Unrated    1165
NR          694
G           405
NC-17        48
PG-13         1
Name: count, dtype: int64

In [14]:
df["Original_Language"].value_counts().head(10)

Original_Language
en    7993
fr     609
it     316
ja     278
es     179
de      99
ko      95
zh      61
cn      55
ru      44
Name: count, dtype: int64

In [15]:
stats["n_english"] = int((df["Original_Language"] == "en").sum())
stats["n_languages"] = int(df["Original_Language"].nunique())

## Vote filter and rating range

The minimum vote count tells us the dataset is already filtered to films
with a real audience — survivorship that limits every conclusion.

In [16]:
stats["total_votes_min"] = int(df["Total_Votes"].min())
stats["audience_rating_min"] = float(df["Audience_Rating"].min())
stats["audience_rating_max"] = float(df["Audience_Rating"].max())
stats["audience_rating_median"] = float(df["Audience_Rating"].median())
df[["Total_Votes", "Audience_Rating", "Popularity_Score"]].describe()

,Total_Votes,Audience_Rating,Popularity_Score
count,10000.000000,10000.000000,10000.000000
mean,2253.043000,6.662549,8.884287
std,3461.315071,0.775403,13.479390
min,367.000000,2.200000,0.006000
25%,563.000000,6.164000,4.522675
50%,999.000000,6.700000,6.365350
75%,2273.000000,7.200000,9.798900
max,41294.000000,9.152000,687.701400


## Skew of the financial and engagement columns

All far above zero — these variables need log scales for any readable plot.

In [17]:
num_cols = ["Budget", "Worldwide_Revenue", "Runtime_Mins", "Audience_Rating", "Total_Votes", "Popularity_Score"]
pd.DataFrame({"skew": df[num_cols].skew(), "kurtosis": df[num_cols].kurt()}).round(2)

,skew,kurtosis
Budget,3.39,18.33
Worldwide_Revenue,5.82,52.61
Runtime_Mins,0.24,7.65
Audience_Rating,-0.33,0.34
Total_Votes,4.06,22.46
Popularity_Score,23.78,906.33


## Secondary dataset (Kaggle)

In [18]:
df2 = load_secondary()
stats["secondary_rows"], stats["secondary_cols"] = df2.shape
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id                 10000 non-null  int64         
 1   title              10000 non-null  str           
 2   genre              9997 non-null   str           
 3   original_language  10000 non-null  str           
 4   overview           9987 non-null   str           
 5   popularity         10000 non-null  float64       
 6   release_date       10000 non-null  datetime64[us]
 7   vote_average       10000 non-null  float64       
 8   vote_count         10000 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), str(4)
memory usage: 3.7 MB


In [19]:
df2.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


Overlap with the primary file, matched on exact title. This is the basis
for the three permitted uses of this file: cross-source validation of
ratings/popularity, recovering the TMDB `id` as a join key, and filling a
handful of missing genres and overviews. The two files are **never**
concatenated — they are overlapping samples of the same population.

In [20]:
overlap_exact = len(set(df["Title"]) & set(df2["title"]))
norm1 = set(df["Title"].str.strip().str.casefold().dropna())
norm2 = set(df2["title"].str.strip().str.casefold().dropna())
overlap_norm = len(norm1 & norm2)
stats["title_overlap_exact"] = overlap_exact
stats["title_overlap_normalised"] = overlap_norm
print(f"unique titles in primary:               {df['Title'].nunique():,}")
print(f"unique titles in secondary:             {df2['title'].nunique():,}")
print(f"titles in both (exact match):           {overlap_exact:,}")
print(f"titles in both (trimmed, case-folded):  {overlap_norm:,}")

unique titles in primary:               9,606
unique titles in secondary:             9,661
titles in both (exact match):           8,276
titles in both (trimmed, case-folded):  8,307


## Schema tables for the report (Chapter 1)

In [21]:
def schema(frame):
    return pd.DataFrame(
        {
            "column": frame.columns,
            "dtype": [str(t) for t in frame.dtypes],
            "non-null": [fmt_int(v) for v in frame.notnull().sum().values],
            "unique": [fmt_int(v) for v in frame.nunique().values],
        }
    )

write_table(schema(df), "tab1_1_primary_schema", align="llrr")
write_table(schema(df2), "tab1_2_secondary_schema", align="llrr")
schema(df)

,column,dtype,non-null,unique
0,Title,str,"10{,}000","9{,}606"
1,Release_Date,datetime64[us],"9{,}997","6{,}215"
2,Release_Year,Int64,"9{,}997",114
3,Budget,int64,"10{,}000",853
4,Worldwide_Revenue,int64,"10{,}000","7{,}072"
5,Runtime_Mins,int64,"10{,}000",206
6,Genres,str,"9{,}998","2{,}101"
7,Content_Rating,str,"10{,}000",8
8,Audience_Rating,float64,"10{,}000","2{,}051"
9,Total_Votes,int64,"10{,}000","3{,}773"


## Write the audit statistics

In [22]:
out = write_stats("ch01_audit", stats)
print(f"{len(stats)} statistics written to {out.relative_to(ROOT)}")
stats

36 statistics written to report\stats\ch01_audit.json


{'primary_rows': 10000,
 'primary_cols': 17,
 'primary_null_release_date': 3,
 'primary_null_genres': 2,
 'primary_null_production_company': 27,
 'primary_null_production_country': 9,
 'zero_budget': 2814,
 'zero_revenue': 2288,
 'both_financials_positive': 6632,
 'budget_but_zero_revenue': 554,
 'mean_budget_with_zeros': 24605346,
 'mean_budget_positive_only': 34240671,
 'budget_1_to_999': 13,
 'revenue_1_to_999': 9,
 'zero_runtime': 1,
 'runtime_max': 367,
 'runtime_median': 104.0,
 'future_films': 92,
 'year_min': 1895,
 'year_max': 2026,
 'exact_duplicate_rows': 4,
 'duplicated_titles': 394,
 'duplicated_title_year': 12,
 'content_rating_unrated': 1165,
 'content_rating_nr': 694,
 'content_rating_stray_pg13': 1,
 'n_english': 7993,
 'n_languages': 44,
 'total_votes_min': 367,
 'audience_rating_min': 2.2,
 'audience_rating_max': 9.152,
 'audience_rating_median': 6.7,
 'secondary_rows': 10000,
 'secondary_cols': 9,
 'title_overlap_exact': 8276,
 'title_overlap_normalised': 8307}

## Audit summary

| Defect | Measured | Handled in |
|---|---|---|
| Zero-coded missing financials | see `zero_budget` / `zero_revenue` above; only `both_financials_positive` rows have both | Ch. 2 |
| Implausible values | budgets/revenues in \$1–999, one zero runtime | Ch. 2 |
| Future films | `future_films` rows dated after 2025 | Ch. 2 (excluded from revenue/ROI) |
| Duplicates | exact duplicate rows; repeated titles are mostly remakes | Ch. 2 (dedupe on title + year) |
| Category mess | `Unrated` vs `NR`, stray `"PG-13 "` | Ch. 2 |
| Multi-label strings | Genres, companies, countries, actors | Ch. 2 / Ch. 4 |
| Heavy right skew | all financial/engagement columns | log scales throughout |
| Survivorship | minimum vote count ≫ 0 | limitations, Ch. 6 |